# Уравнение Бюргерса

## Базовый уровень

Решаем невязкое уравнение Бюргерса:

$$\frac{\partial u}{\partial t} + u \frac{\partial u}{\partial x} = \nu \frac{\partial^2 u}{\partial x^2}.$$

Или в консервативной форме:

$$\frac{\partial u}{\partial t} + \frac{F(u)}{\partial x} = 0, \quad F(u) = \frac{u^2}{2}.$$

Upwind схема для невязкого уравнения Бюргерса ($\nu = 0$)

\begin{equation*}
u^{n+1}_i = 
\begin{cases}
u^n_i - \frac{\Delta t}{\Delta x} \left(F(u^n_{i+1}) - F(u^n_i)\right), \quad u^n_i > 0,\\
u^n_i - \frac{\Delta t}{\Delta x} \left(F(u^n_{i}) - F(u^n_{i-1})\right), \quad u^n_i < 0.
\end{cases}
\end{equation*}

Схема Лакса-Вендроффа:

$$u^{n+1}_i = u^n_i - \frac{\Delta t}{2 \Delta x} \left[F(u^n_{i+1}) - F(u^n_{i-1})\right] + \frac{\Delta t^2}{2 \Delta x^2}\left[A_{i+1/2}\left(F(u^n_{i+1}) - F(u^n_i)\right) - A_{i-1/2}\left(F(u^n_i) - F(u^n_{i-1})\right)\right],$$

где $A_{i \pm 1/2}$ -- якобиан $F(u)$ в точке $\frac{1}{2}(u^n_i + u^n_{i \pm 1})$:

$$\frac{\partial F(u)}{\partial u} = \frac{\partial}{\partial u}\left(\frac{u^2}{2}\right) = u.$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as anim

In [ ]:
def flux(u):
    return 0.5 * u ** 2

def upwind_sim(u, dt, dx, time_steps):
    for i in range(time_steps):
        f = flux(u[i, :])
        u[i+1, :] = u[i, :] - (dt / dx) * np.where(u[i, :] > 0, f - np.roll(f, 1), np.roll(f, -1) - f)
    return u

def jacobian(u):
    return u

def lax_wendroff_sim(u, dt, dx, time_steps):
    for i in range(time_steps):
        f = flux(u[i, :])
        f_plus  = np.roll(f, -1)
        f_minus = np.roll(f, 1)
        a_plus  = 0.5 * (u[i, :] + np.roll(u[i, :], -1))
        a_minus = 0.5 * (u[i, :] + np.roll(u[i, :], 1))
        u[i+1, :] = u[i, :] - (dt / (2.0 * dx)) * (f_plus - f_minus) + (dt ** 2) / (2.0 * dx ** 2) * (a_plus * (f_plus - f) - a_minus * (f - f_minus))
    return u

Решаем уравнение в области $[0, 1]$ с начальными условиями $u(0, x) = \sin(2 \pi x)$ и периодическими граничными условиями $u(t, 0) = u(t, 1)$.

In [ ]:
def initial_condition(x):
    return np.sin(2*np.pi*x)

In [ ]:
n = 100
x = np.linspace(0.0, 1.0, n)
dx = x[1] - x[0]
dt = 0.001
time_steps = 250

u_uw = np.zeros((time_steps + 1, n))
u_uw[0, :] = initial_condition(x)
u_lw = np.zeros((time_steps + 1, n))
u_lw[0, :] = initial_condition(x)

In [ ]:
u_uw = upwind_sim(u_uw, dt, dx, time_steps)
u_lw = lax_wendroff_sim(u_lw, dt, dx, time_steps)

In [ ]:
fig, ax = plt.subplots()
ax.set_xlim(0.0, 1.0)
ax.grid(True)
ax.set_xlabel('x')
ax.set_ylabel('u(t, x)')
ax.set_title('Невязкое уравнение Бюргерса')

line1, = ax.plot(x, u_uw[0, :], c='blue', alpha=0.5)
line2, = ax.plot(x, u_lw[0, :], c='red', alpha=0.5)

ax.legend(['Upwind', 'Lax-Wendroff'])

def animate(i, x, u_uw, u_lw):
    line1.set_data(x, u_uw[i, :])
    line2.set_data(x, u_lw[i, :])
    return line1, line2,

ani = anim.FuncAnimation(fig, animate, frames=time_steps + 1, interval=100, fargs=(x, u_uw, u_lw,))

from IPython.display import HTML
HTML(ani.to_jshtml())

Со временем образуется ударная волна. Вблизи разрыва у схемы Лакса-Вндроффа наблюдается диссипация.

## Продвинутый уровень

Решаем вязкое уравнение Бюргерса ($\nu \neq 0$):

$$\frac{\partial u}{\partial t} + u \frac{\partial u}{\partial x} = \nu \frac{\partial^2 u}{\partial x^2}.$$

К схемам добавляется вязкостный член, который дискретизируется следующим образом

$$\nu \frac{u^n_{i-1} - 2 u^n_i + u^n_{i+1}}{h^2}.$$

Upwind схема для вязкого уравнения Бюргерса ($\nu \neq 0$):

\begin{equation*}
u^{n+1}_i = 
\begin{cases}
u^n_i - \frac{\Delta t}{\Delta x} \left(F(u^n_{i+1}) - F(u^n_i)\right) + \nu \frac{u^n_{i-1} - 2 u^n_i + u^n_{i+1}}{h^2}, \quad u^n_i > 0,\\
u^n_i - \frac{\Delta t}{\Delta x} \left(F(u^n_{i}) - F(u^n_{i-1})\right) + \nu \frac{u^n_{i-1} - 2 u^n_i + u^n_{i+1}}{h^2}, \quad u^n_i < 0.
\end{cases}
\end{equation*}

Схема Лакса-Вендроффа:

$$u^{n+1}_i = u^n_i - \frac{\Delta t}{2 \Delta x} \left[F(u^n_{i+1}) - F(u^n_{i-1})\right] + \frac{\Delta t^2}{2 \Delta x^2}\left[A_{i+1/2}\left(F(u^n_{i+1}) - F(u^n_i)\right) - A_{i-1/2}\left(F(u^n_i) - F(u^n_{i-1})\right)\right] + \nu \frac{\Delta t}{\Delta x^2} \left(u^n_{i-1} - 2 u^n_i + u^n_{i+1}\right),$$

где $A_{i \pm 1/2}$ -- якобиан $F(u)$ в точке $\frac{1}{2}(u^n_i + u^n_{i \pm 1})$.

In [ ]:
def upwind_viscous_sim(u, dt, dx, time_steps, nu = 0.1, periodic_bc = True):
    if periodic_bc:
        for i in range(time_steps):
            f = flux(u[i, :])
            u[i+1, :] = u[i, :] - (dt / dx) * np.where(u[i, :] > 0, f - np.roll(f, 1), np.roll(f, -1) - f) + nu * dt / dx ** 2 * (np.roll(u[i, :], -1) - 2 * u[i, :] + np.roll(u[i, :], 1))
    else:
        for i in range(time_steps):
            f = flux(u[i, :])
            u_plus = np.roll(u[i, :], -1)
            u_minus = np.roll(u[i, :], 1)
            u_plus[-1] = u_plus[-2]
            u_minus[0] = u_minus[1]
            f_plus = flux(u_plus)
            f_minus = flux(u_minus)
            u[i+1, :] = u[i, :] - (dt / dx) * np.where(u[i, :] > 0, f - f_minus, f_plus - f) + nu * dt / dx ** 2 * (u_plus - 2 * u[i, :] + u_minus)
    return u

def lax_wendroff_viscous_sim(u, dt, dx, time_steps, nu = 0.01, periodic_bc = True):
    if periodic_bc:
        for i in range(time_steps):
            f = flux(u[i, :])

            f_plus  = np.roll(f, -1)
            f_minus = np.roll(f, 1)
            a_plus  = 0.5 * (u[i, :] + np.roll(u[i, :], -1))
            a_minus = 0.5 * (u[i, :] + np.roll(u[i, :], 1))
            u[i+1, :] = u[i, :] - (dt / (2.0 * dx)) * (f_plus - f_minus) + (dt**2) / (2.0 * dx**2) * (a_plus * (f_plus - f) - a_minus * (f - f_minus)) + nu * dt / dx ** 2 * (np.roll(u[i, :], -1) - 2 * u[i, :] + np.roll(u[i, :], 1))
    else:
        for i in range(time_steps):
            f = flux(u[i, :])
            u_plus = np.roll(u[i, :], -1)
            u_minus = np.roll(u[i, :], 1)
            u_plus[-1] = u_plus[-2]
            u_minus[0] = u_minus[1]
            f_plus = flux(u_plus)
            f_minus = flux(u_minus)
            a_plus  = 0.5 * (u[i, :] + u_plus)
            a_minus = 0.5 * (u[i, :] + u_minus)
            u[i+1, :] = u[i, :] - (dt / (2.0 * dx)) * (f_plus - f_minus) + (dt**2) / (2.0 * dx**2) * (a_plus * (f_plus - f) - a_minus * (f - f_minus)) + nu * dt / dx ** 2 * (u_plus - 2 * u[i, :] + u_minus)
    return u

In [ ]:
n = 50
x = np.linspace(0.0, 1.0, n)
dx = x[1] - x[0]
dt = 0.005
time_steps = 300
T = dt * time_steps
nu = 0.02

u1 = np.zeros((time_steps + 1, n))
u1[0, :] = initial_condition(x)
u2 = np.zeros((time_steps + 1, n))
u2[0, :] = initial_condition(x)
u3 = np.zeros((time_steps + 1, n))
u3[0, :] = initial_condition(x)
u4 = np.zeros((time_steps + 1, n))
u4[0, :] = initial_condition(x)

In [ ]:
nu_array = [0.00025, 0.005, 0.01, 0.02]
u1 = upwind_viscous_sim(u1, dt, dx, time_steps, nu=nu_array[0])
u2 = upwind_viscous_sim(u2, dt, dx, time_steps, nu=nu_array[1])
u3 = upwind_viscous_sim(u3, dt, dx, time_steps, nu=nu_array[2])
u4 = upwind_viscous_sim(u4, dt, dx, time_steps, nu=nu_array[3])

In [ ]:
fig, ax = plt.subplots()
ax.set_xlim(0.0, 1.0)
ax.grid(True)
ax.set_xlabel('x')
ax.set_ylabel('u(t, x)')
ax.set_title('Вязкое уравнение Бюргерса, T = {}'.format(T))

line1, = ax.plot(x, u1[0, :], c='blue', alpha=0.5)
line2, = ax.plot(x, u2[0, :], c='red', alpha=0.5)
line3, = ax.plot(x, u3[0, :], alpha=0.5)
line4, = ax.plot(x, u4[0, :], alpha=0.5)

ax.legend([r'\nu = 0.0025', r'\nu = 0.005', r'\nu = 0.01', r'\nu = 0.02'])

def animate(i, x, u1, u2, u3, u4):
    line1.set_data(x, u1[i, :])
    line2.set_data(x, u2[i, :])
    line3.set_data(x, u3[i, :])
    line4.set_data(x, u4[i, :])
    return line1, line2, line3, line4,

ani = anim.FuncAnimation(fig, animate, frames=time_steps // 5, interval=100, fargs=(x, u1[::5], u2[::5], u3[::5], u4[::5],))

from IPython.display import HTML
HTML(ani.to_jshtml())

В отличие от невязкого уравнения Бюргерса сильного разрыва не образуется (за счёт диффузионного члена).

Сравним численное решение для стационарной ударной волны с аналитическим решением

$$u(x,t) = c - U \tanh\left(U \frac{x - c t}{2\nu}\right),$$

где $c = \frac{f^+ + f^-}{2}$, $U = \frac{f^+ - f^-}{2}$.

In [ ]:
def exact_solution(x, t, f_minus, f_plus, nu):
    c = (f_plus + f_minus) / 2
    U = (f_plus - f_minus) / 2
    return c - U * np.tanh(U * (x - c * t) / 2 / nu)

In [ ]:
n = 100
x = np.linspace(-0.2, 0.8, n)
dx = x[1] - x[0]
dt = 0.0025
time_steps = 250
T = dt * time_steps
nu = 0.005

u1 = np.zeros((time_steps + 1, n))
u1[0, :] = exact_solution(x, 0.0, 0, 2, nu)
u2 = np.zeros((time_steps + 1, n))
u2[0, :] = exact_solution(x, 0.0, 0, 2, nu)

In [ ]:
u1 = upwind_viscous_sim(u1, dt, dx, time_steps, nu, False)
u2 = lax_wendroff_viscous_sim(u2, dt, dx, time_steps, nu, False)

In [ ]:
fig, ax = plt.subplots()
ax.set_xlim(-0.2, 0.8)
ax.set_ylim(-0.1, 2.2)
ax.grid(True)
ax.set_xlabel('x')
ax.set_ylabel('u(t, x)')
ax.set_title('Вязкое уравнение Бюргерса')

line1, = ax.plot(x, exact_solution(x, 0, 0, 2, nu), c='blue', alpha=0.5)
line2, = ax.plot(x, u1[0, :], c='green', alpha=0.5)
line3, = ax.plot(x, u2[0, :], c='red', alpha=0.5)

ax.legend(['Exact', 'Upwind', 'Lax-Wendroff'])

def animate(i, u1, u2):
    line1.set_data(x, exact_solution(x, 10*i*dt, 0, 2, nu))
    line2.set_data(x, u1[i, :])
    line3.set_data(x, u2[i, :])
    return line1, line2, line3,

ani = anim.FuncAnimation(fig, animate, frames=time_steps // 10, interval=200, fargs=(u1[::10], u2[::10],))

from IPython.display import HTML
HTML(ani.to_jshtml())

Схема Лакса-Вендроффа имеет большую точность по сравнению с Upwind-схемой.

Оценим зависимость ширины ударного фронта от коэффициента вязкости $\nu$.

In [ ]:
nu_len = 10
nu_array = np.linspace(0.001, 0.03, nu_len)

n = 100
x = np.linspace(-0.2, 0.8, n)
dx = x[1] - x[0]
dt = 0.001
time_steps = 300
T = dt * time_steps
nu = 0.01

u = np.zeros((nu_len, time_steps + 1, n))

In [ ]:
for i in range(nu_len):
    u[i, 0, :] = exact_solution(x, 0.0, 0.0, 2.0, nu_array[i])
    u[i, :, :] = upwind_viscous_sim(u[i, :, :], dt, dx, time_steps, nu_array[i], False)

In [ ]:
fig, ax = plt.subplots()
ax.set_xlim(-0.2, 0.8)
ax.set_ylim(0.0, 2.1)
ax.grid(True)
ax.set_xlabel('x')
ax.set_ylabel('u(t, x)')
ax.set_title('Вязкое уравнение Бюргерса')

line = []
for i in range(nu_len):
    line.append(ax.plot(x, u[i, 0, :], alpha=0.5)[0])
    line[-1].set_label('nu = {:.3f}'.format(nu_array[i]))
ax.legend()

def animate(i, u):
    for j in range(nu_len):
        line[j].set_data(x, u[j, i, :])
    return *line,

ani = anim.FuncAnimation(fig, animate, frames=time_steps // 10, interval=100, fargs=(u[:,::10,:],))

from IPython.display import HTML
HTML(ani.to_jshtml())

In [ ]:
def shock_width_90_10(u, x):
    u_min, u_max = np.min(u), np.max(u)
    u_10 = u_min + 0.1 * (u_max - u_min)
    u_90 = u_min + 0.9 * (u_max - u_min)

    x10_candidates = np.interp(u_10, u, x)
    x90_candidates = np.interp(u_90, u[::-1], x[::-1])

    width = abs(x90_candidates - x10_candidates)
    return width

In [ ]:
widths = []
for i in range(nu_len):
    widths.append(shock_width_90_10(u[i, -1, :], x))

In [ ]:
plt.plot(nu_array, widths)
plt.xlabel('nu')
plt.ylabel('Ширина фронта')
plt.show()

Ширина фронта ударной волны линейно зависит от коэффициента вязкости $\nu$.